In [6]:
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 42
D_MODEL = 128
N_HEADS = 4
N_LAYERS = 2
MAX_LEN = 32
BATCH_SIZE = 64
EPOCHS = 15
LR = 1e-3
WARM_FRAC = 0.1
MLM_PROB = 0.2
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

PAD, UNK, MASK, CLS, SEP = "[PAD]", "[UNK]", "[MASK]", "[CLS]", "[SEP]"
SPECIAL_TOKENS = [PAD, UNK, MASK, CLS, SEP]

# --------------------------------------------------------------------------- #
# Toy corpus (synthetic, so the model has real structure to learn)
# --------------------------------------------------------------------------- #

def build_corpus(n_sentences=4000):
    """Animals eat specific foods and live in specific places."""
    facts = {
        "cat": ("fish", "house"),
        "dog": ("bone", "yard"),
        "cow": ("grass", "farm"),
        "rabbit": ("carrot", "burrow"),
        "monkey": ("banana", "jungle"),
        "bird": ("seeds", "nest"),
    }
    adjectives = ["small", "happy", "hungry", "quiet", "big"]
    templates = [
        "the {adj} {animal} eats {food}",
        "the {animal} likes {food} and lives in the {place}",
        "every day the {adj} {animal} finds {food}",
        "the {animal} sleeps in the {place}",
        "a {adj} {animal} lives in the {place} and eats {food}",
    ]
    corpus = []

    for _ in range(n_sentences):
        animal, (food, place) = random.choice(list(facts.items()))
        sent = random.choice(templates).format(
                adj=random.choice(adjectives), animal=animal, food=food, place=place
            )
        corpus.append(sent)
        return corpus

        


In [3]:
class Vocab:
    def __init__(self, sentences):
        words = sorted({w for s in sentences for w in s.split()})
        self.itos = SPECIAL_TOKENS + words
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.pad_id = self.stoi[PAD]
        self.unk_id = self.stoi[UNK]
        self.mask_id = self.stoi[MASK]
        self.cls_id = self.stoi[CLS]
        self.sep_id = self.stoi[SEP]
        self.num_special = len(SPECIAL_TOKENS)

    def __len__(self):
        return len(self.itos)

    def encode(self, sentence, add_special=True):
        ids = [self.stoi.get(w, self.unk_id) for w in sentence.split()]
        if add_special:
            ids = [self.cls_id] + ids + [self.sep_id]
        return ids[:MAX_LEN]

    def decode(self, ids):
        return " ".join(self.itos[i] for i in ids)


In [7]:
# --------------------------------------------------------------------------- #
# Masking
# --------------------------------------------------------------------------- #
def mask_tokens(input_ids, vocab, mlm_prob=MLM_PROB):
    """
        BERT-style dynamic masking. Of the selected positions:
          80% -> [MASK], 10% -> random token, 10% -> left unchanged.
        Returns (masked_input_ids, labels) where labels = -100 at unselected positions.
        """
    input_ids = input_ids.clone()
    labels = input_ids.clone()
    # Never select special tokens or padding
    special = (
        (input_ids == vocab.pad_id)
        | (input_ids == vocab.cls_id)
        | (input_ids == vocab.sep_id)
    )
    prob = torch.full(labels.shape,mlm_prob)
    prob = torch.masked_fill(special, 0.0)
    selected = torch.bernoulli(prob).bool()
    labels[~selected] = -100  # loss only on selected positions
    
    # 10%: replace with a random (non-special) token; remaining 10% unchanged
    to_random = (
        torch.bernoulli(torch.full(labels.shape, 0.5)).bool() & selected & ~to_mask
    )
    random_ids = torch.randint(vocab.num_special, len(vocab), labels.shape)
    input_ids[to_random] = random_ids[to_random]

    return input_ids, labels


# --------------------------------------------------------------------------- #
# Dataset / DataLoader
# --------------------------------------------------------------------------- #
class TextDataset(Dataset):
    def __init__(self, sentences, vocab):
        self.examples = [vocab.encode(s) for s in sentences]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def make_collate_fn(vocab, apply_masking=True):
    def collate(batch):
        max_len = max(len(x) for x in batch)
        input_ids = torch.full((len(batch), max_len), vocab.pad_id, dtype=torch.long)
        for i, ids in enumerate(batch):
            input_ids[i, : len(ids)] = torch.tensor(ids)
        attention_mask = (input_ids != vocab.pad_id).long()  # 1 = real token

        if apply_masking:
            masked_ids, labels = mask_tokens(input_ids, vocab)
        else:
            masked_ids, labels = input_ids, input_ids.clone()
        return masked_ids, attention_mask, labels

    return collate

In [ ]:
# --------------------------------------------------------------------------- #
# Model
# --------------------------------------------------------------------------- #
class MLMHead(nn.Module):
    def __init__(self,d_model, vocab_size):
        super().__init__()
        self.dense = nn.Linear(d_model, d_model)
        self.activation = nn.GELU()
        self.layer_norm = nn.LayerNorm(d_model)
        self.decoder = nn.Linear(d_model, vocab_size)
    def forward(self, hidden_states):
        x = self.dense(hidden_states)
        x = self.activation(x)
        x = self.layer_norm(x)
        return self.decoder(x)

In [ ]:
class TinyMLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=128,
        n_heads=4,
        n_layers=2,
        max_len=128,
        tie_weights=True,
    ):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=0.1,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=n_layers, enable_nested_tensor=False
        )
        self.mlm_head = MLMHead(d_model, vocab_size)
        # if tie_weights:  # share input embedding and output projection
        #     self.mlm_head.decoder.weight = self.token_emb.weight

        # self.apply(self._init_weights)

    def forward(self, input_ids, attention_mask=None):
            """
            input_ids:      (B, L) long
            attention_mask: (B, L) with 1 = real token, 0 = padding (optional)
            returns logits: (B, L, vocab_size)
            """
            _, seq_len = input_ids.shape
            positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
    
            x = self.token_emb(input_ids) + self.pos_emb(positions)
            x = self.layer_norm(x)
            x = self.dropout(x)
    
            pad_mask = None if attention_mask is None else (attention_mask == 0)
            x = self.encoder(x, src_key_padding_mask=pad_mask)
    
            return self.mlm_head(x)

def mlm_loss(logits, labels):
    return F.cross_entropy(
        logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100
    )


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, total_correct, total_masked = 0.0, 0, 0
    for input_ids, attn, labels in loader:
        input_ids, attn, labels = (t.to(DEVICE) for t in (input_ids, attn, labels))
        logits = model(input_ids, attn)
        sel = labels != -100
        n = sel.sum().item()
        if n == 0:
            continue
        total_loss += mlm_loss(logits, labels).item() * n
        total_correct += (logits.argmax(-1)[sel] == labels[sel]).sum().item()
        total_masked += n
    return total_loss / total_masked, total_correct / total_masked


